# Efficient LR-QAOA vs WalkSAT benchmark (BM24)

Aligned with `bm24_qaoa_sim.py` and **`train_lr_notebook_protocol.py` (v3)** — training minimises slope of `mean(ln(1/p_succ))` vs `n` (smoother for COBYLA). **Eval** still uses `ln(median(1/p_succ))` vs `n`.

**Unlike `sweep_lr_depth_until_win.py`, this notebook always runs every depth in `CFG["depths"]` — it does not stop early when LR beats WalkSAT.**

1. **SAT benchmark set built once** — `H_diag` and Numba clause arrays cached per instance.
2. **WalkSAT + WalkSATlm run once** (Numba) — not repeated every QAOA depth.
3. **Training set at `n_train` built once** (100 instances) — reused for every depth.
4. **Per depth:** v3 training — slope of `mean(ln(1/p))` vs `n`, anti-regression vs warm-start, top‑K grid, multi-restart COBYLA.
5. **Eval guard:** reject depth if `median(1/p)` at `n_min` or `n_max` exceeds 10× the previous depth; **re-train** up to `eval_train_retries` times, then keep previous angles.
6. **Proxy set** at `n_train, n_train+2, n_train+4` (`proxy_n_span=4`, step 2) with 30 instances/n.

**Eval:** log₂ slope of **median(1/p_succ)** vs `n` (lower = milder growth; beat WalkSAT when LR slope < WalkSAT line).

**Plot:** depths where training/eval could not accept new angles are marked with **×** (prior angles used).


**Classical parameters in `CFG`:**
- `walksat_p_noise` — random-walk probability for WalkSAT (**0.5**).
- `walksatlm_p_noise` — random-walk probability for WalkSATlm (**0.15**).
- `walksatlm_w1`, `walksatlm_w2` — WalkSATlm linear-make weights (default 6 and 5).

In [ ]:
# --- Configuration (edit here) ---
from pathlib import Path
import sys


def _notebook_bootstrap() -> None:
    """Put repo root + phasecraft on sys.path (works from repo root, phasecraft/, or notebooks/)."""
    here = Path.cwd().resolve()
    repo = here
    phasecraft = here / "phasecraft"
    for base in (here, *here.parents):
        pc = base / "phasecraft"
        if (pc / "lib" / "paths.py").is_file():
            repo, phasecraft = base, pc
            break
        if (base / "lib" / "paths.py").is_file() and base.name == "phasecraft":
            repo, phasecraft = base.parent, base
            break
    for p in (repo, phasecraft):
        s = str(p)
        if s not in sys.path:
            sys.path.insert(0, s)


_notebook_bootstrap()

CFG = {
    "k": 8,
    "r": 176.54,
    "seed": 27,
    "train_n": 12,
    "train_size": 100,
    "n_min": 12,
    "n_max": 20,
    "test_size": 200,
    "depths": [2,5,8, 10, 15, 20], #list(range(2, 30)),  # or e.g. ...
    "skip_grid": False,
    "skip_grid_if_warm_start": False,  # depth>=2: COBYLA from previous (dg,db), no grid
    "cobyla_maxiter": 160, #200,
    "cobyla_restarts": 8,
    "cobyla_perturb_scale": 0.2,
    "grid_top_k": 5,
    # v2: objective is slope of ln(median(1/p)) vs n; w_mean/w_slope ignored
    "proxy_size_per_n": 30, #50,
    "proxy_n_span": 4,            # with step=2 -> n in [12, 14, 16]
    "eval_axis": "runtime",       # "runtime" (cost / 1/p) | "success" (p_succ)
    "eval_aggregation": "median", # "median" | "mean" — statistic on eval axis
    "legacy_objective": False,    # True: use train_lr_notebook_protocol_legacy.py
    "walksat_p_noise": 0.5,
    "walksatlm_p_noise": 0.15,
    "walksatlm_w1": 6,            # WalkSATlm only: weight on make-1 clauses when tie-breaking
    "walksatlm_w2": 5,            # WalkSATlm only: weight on make-2 clauses
    "annotate_first_win": False,  # plot only; never stops the depth loop early
    "eval_train_retries": 3,  # re-train after eval regression before prior angles
    "max_flips": 100_000,
    "lr_beta_schedule": "decreasing",
    "output_dir": None,  # set below to canonical bm24_runs_dir()
    "angle_log": None,  # set below: legacy vs v2/v3 angle log (CLI --angle-log)
    # Set to resume after kernel restart / crash (see recovery cell below).
    "recover_from": None,
    "run_stem": "06-01-retry-training-fix-test",  # keep output names when recovering; None = new timestamp
}
from bm24_run_io import normalize_eval_aggregation, normalize_eval_axis
from phasecraft.lib.paths import (
    bm24_runs_dir,
    lr_train_optimal_angles_legacy_path,
    lr_train_optimal_angles_v2_path,
)

CFG["eval_axis"] = normalize_eval_axis(CFG.get("eval_axis", "runtime"))
CFG["eval_aggregation"] = normalize_eval_aggregation(CFG.get("eval_aggregation", "median"))
if CFG.get("legacy_objective"):
    CFG["eval_axis"] = "success"
if CFG.get("output_dir") is None:
    CFG["output_dir"] = bm24_runs_dir()
else:
    CFG["output_dir"] = Path(CFG["output_dir"])
CFG["output_dir"].mkdir(parents=True, exist_ok=True)
CFG["angle_log"] = (
    lr_train_optimal_angles_legacy_path()
    if CFG.get("legacy_objective")
    else lr_train_optimal_angles_v2_path()
)
if CFG.get("recover_from") is not None:
    CFG["recover_from"] = Path(CFG["recover_from"])
print(CFG)

In [ ]:
import json
import time
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
from numba import njit
from scipy.stats import linregress
from tqdm.auto import tqdm

# Paths set in CFG cell via _notebook_bootstrap()
_notebook_bootstrap()

from bm24_qaoa_sim import (
    build_h_diagonal,
    generate_random_clause,
    generate_random_formula,
    make_lr_angles,
    per_instance_success_probability,
    run_qaoa,
)
from train_lr_notebook_protocol import (
    DEFAULT_EVAL_RUNTIME_REGRESSION_FACTOR,
    DEFAULT_EVAL_TRAIN_RETRIES,
    eval_median_runtime_reject,
    generate_training_h_diagonals,
    generate_training_h_diagonals_multi_n,
    proxy_n_values_for_training,
    run_train_eval_with_retries,
    train_lr_grid_search_bm24,
)
if CFG.get("legacy_objective", False):
    from train_lr_notebook_protocol_legacy import (
        train_lr_grid_search_bm24 as train_lr_grid_search_legacy,
    )

K = int(CFG["k"])
LN2 = float(np.log(2.0))

In [ ]:
# --- Numba classical solvers (notebook-accelerated; k literals per clause) ---

@njit(cache=True)
def fast_walksat_solver(n, c_vars, c_signs, max_flips, p_noise):
    assignment = np.random.randint(0, 2, n)
    m, k_sat = c_vars.shape
    unsat_buffer = np.empty(m, dtype=np.int32)
    for flip in range(max_flips):
        unsat_count = 0
        for i in range(m):
            is_sat = False
            for kk in range(k_sat):
                v = c_vars[i, kk]
                if assignment[v] == c_signs[i, kk]:
                    is_sat = True
                    break
            if not is_sat:
                unsat_buffer[unsat_count] = i
                unsat_count += 1
        if unsat_count == 0:
            return flip + 1
        target_c_idx = unsat_buffer[np.random.randint(0, unsat_count)]
        if np.random.random() < p_noise:
            var_to_flip = c_vars[target_c_idx, np.random.randint(0, k_sat)]
        else:
            best_var = -1
            min_breaks = 999999
            for kk in range(k_sat):
                candidate_var = c_vars[target_c_idx, kk]
                assignment[candidate_var] = 1 - assignment[candidate_var]
                current_breaks = 0
                for i_scan in range(m):
                    c_sat = False
                    for kk2 in range(k_sat):
                        v_scan = c_vars[i_scan, kk2]
                        if assignment[v_scan] == c_signs[i_scan, kk2]:
                            c_sat = True
                            break
                    if not c_sat:
                        current_breaks += 1
                if current_breaks < min_breaks:
                    min_breaks = current_breaks
                    best_var = candidate_var
                assignment[candidate_var] = 1 - assignment[candidate_var]
            var_to_flip = best_var
        assignment[var_to_flip] = 1 - assignment[var_to_flip]
    return max_flips


@njit(cache=True)
def walksatlm_paper_kernel(n, c_vars, c_signs, max_flips, p_noise, w1, w2):
    m = c_vars.shape[0]
    k_sat = c_vars.shape[1]
    degrees = np.zeros(n, dtype=np.int32)
    for i in range(m):
        for kk in range(k_sat):
            degrees[c_vars[i, kk]] += 1
    max_degree = 0
    for i in range(n):
        if degrees[i] > max_degree:
            max_degree = degrees[i]
    adj_indices = np.full((n, max_degree), -1, dtype=np.int32)
    adj_signs = np.full((n, max_degree), -1, dtype=np.int32)
    current_fill = np.zeros(n, dtype=np.int32)
    for i in range(m):
        for kk in range(k_sat):
            v = c_vars[i, kk]
            s = c_signs[i, kk]
            pos = current_fill[v]
            adj_indices[v, pos] = i
            adj_signs[v, pos] = s
            current_fill[v] += 1
    assignment = np.random.randint(0, 2, n)
    num_true_lits = np.zeros(m, dtype=np.int32)
    for i in range(m):
        count = 0
        for kk in range(k_sat):
            if assignment[c_vars[i, kk]] == c_signs[i, kk]:
                count += 1
        num_true_lits[i] = count
    unsat_buffer = np.empty(m, dtype=np.int32)
    for flip in range(1, max_flips + 1):
        unsat_count = 0
        for i in range(m):
            if num_true_lits[i] == 0:
                unsat_buffer[unsat_count] = i
                unsat_count += 1
        if unsat_count == 0:
            return flip
        target_c_idx = unsat_buffer[np.random.randint(0, unsat_count)]
        candidates = c_vars[target_c_idx]
        cand_breaks = np.zeros(k_sat, dtype=np.int32)
        cand_lmakes = np.zeros(k_sat, dtype=np.int32)
        has_zero_break = False
        for kk in range(k_sat):
            var = candidates[kk]
            current_break = 0
            make_1 = 0
            make_2 = 0
            deg = current_fill[var]
            for idx in range(deg):
                c_idx = adj_indices[var, idx]
                s = adj_signs[var, idx]
                lit_count = num_true_lits[c_idx]
                if assignment[var] == s:
                    if lit_count == 1:
                        current_break += 1
                else:
                    if lit_count == 0:
                        make_1 += 1
                    elif lit_count == 1:
                        make_2 += 1
            cand_breaks[kk] = current_break
            cand_lmakes[kk] = w1 * make_1 + w2 * make_2
            if current_break == 0:
                has_zero_break = True
        best_var = -1
        if has_zero_break:
            best_val = -1e9
            for kk in range(k_sat):
                if cand_breaks[kk] == 0:
                    score = cand_lmakes[kk]
                    if score > best_val:
                        best_val = score
                        best_var = candidates[kk]
                    elif score == best_val and np.random.random() < 0.5:
                        best_var = candidates[kk]
        elif np.random.random() < p_noise:
            best_var = candidates[np.random.randint(0, k_sat)]
        else:
            min_b = 999999
            max_l = -999999
            for kk in range(k_sat):
                b = cand_breaks[kk]
                lmk = cand_lmakes[kk]
                if b < min_b:
                    min_b = b
                    max_l = lmk
                    best_var = candidates[kk]
                elif b == min_b:
                    if lmk > max_l:
                        max_l = lmk
                        best_var = candidates[kk]
                    elif lmk == max_l and np.random.random() < 0.5:
                        best_var = candidates[kk]
        assignment[best_var] = 1 - assignment[best_var]
        deg = current_fill[best_var]
        for idx in range(deg):
            c_idx = adj_indices[best_var, idx]
            s = adj_signs[best_var, idx]
            if assignment[best_var] == s:
                num_true_lits[c_idx] += 1
            else:
                num_true_lits[c_idx] -= 1
    return max_flips

In [ ]:
def clauses_to_numba_arrays(clauses, k: int):
    """BM24 (var, is_negated) -> Numba signs with literal true iff assignment == sign."""
    m = len(clauses)
    c_vars = np.zeros((m, k), dtype=np.int32)
    c_signs = np.zeros((m, k), dtype=np.int32)
    for i, clause in enumerate(clauses):
        for j, (var, is_negated) in enumerate(clause):
            c_vars[i, j] = int(var)
            c_signs[i, j] = 1 - int(bool(is_negated))
    return c_vars, c_signs


def generate_benchmark_dataset_cached(
    n_values,
    k: int,
    r: float,
    test_size: int,
    base_seed: int,
) -> Dict[int, List[dict]]:
    """SAT-filtered instances with cached H_diag and Numba clause arrays."""
    dataset: Dict[int, List[dict]] = {}
    for n in n_values:
        n = int(n)
        accepted = 0
        trial = 0
        instances = []
        pbar = tqdm(total=test_size, desc=f"dataset n={n}", leave=False)
        while accepted < test_size:
            if trial > 200_000:
                raise RuntimeError(f"too many rejections at n={n}")
            ss = np.random.SeedSequence([int(base_seed), n, accepted, trial])
            rng = np.random.default_rng(ss)
            lam = float(r) * n
            m_clauses = max(1, int(rng.poisson(lam)))
            clauses = [generate_random_clause(n, k, rng) for _ in range(m_clauses)]
            h_diag = build_h_diagonal(clauses, n)
            if not np.any(h_diag == 0):
                trial += 1
                continue
            c_vars, c_signs = clauses_to_numba_arrays(clauses, k)
            instances.append({
                "clauses": clauses,
                "h_diag": h_diag,
                "c_vars": c_vars,
                "c_signs": c_signs,
            })
            accepted += 1
            trial += 1
            pbar.update(1)
        pbar.close()
        dataset[n] = instances
    return dataset


def _seed_for_instance(base_seed: int, n: int, idx: int) -> int:
    return int(np.random.SeedSequence([base_seed, n, idx]).generate_state(1)[0])


def evaluate_classical_once(dataset, cfg) -> dict:
    """Run WalkSAT + WalkSATlm once; return median flips per n."""
    p_ws = float(cfg.get("walksat_p_noise", cfg.get("p_noise", 0.5)))
    p_lm = float(cfg.get("walksatlm_p_noise", 0.15))
    max_flips = int(cfg["max_flips"])
    w1, w2 = int(cfg["walksatlm_w1"]), int(cfg["walksatlm_w2"])
    base_seed = int(cfg["seed"])
    out = {"walksat": {}, "walksatlm": {}}
    for n in sorted(dataset.keys()):
        ws_flips, lm_flips = [], []
        for idx, inst in enumerate(tqdm(dataset[n], desc=f"classical n={n}", leave=False)):
            seed = _seed_for_instance(base_seed, n, idx)
            np.random.seed(seed)
            ws_flips.append(
                int(fast_walksat_solver(n, inst["c_vars"], inst["c_signs"], max_flips, p_ws))
            )
            np.random.seed(seed)
            lm_flips.append(
                int(walksatlm_paper_kernel(n, inst["c_vars"], inst["c_signs"], max_flips, p_lm, w1, w2))
            )
        out["walksat"][n] = float(np.median(ws_flips))
        out["walksatlm"][n] = float(np.median(lm_flips))
    return out


def fit_log2_slope(n_values, y_values) -> float:
    n_arr = np.asarray(n_values, dtype=float)
    y_arr = np.asarray(y_values, dtype=float)
    mask = np.isfinite(y_arr) & (y_arr > 0)
    if mask.sum() < 2:
        return float("nan")
    slope_nat = linregress(n_arr[mask], np.log(y_arr[mask])).slope
    return float(slope_nat / LN2)


def evaluate_lr_qaoa_depth(dataset, n_values, betas, gammas, eps: float = 1e-300) -> dict:
    """Median(1/p_succ) per n using cached H_diag (BM24 run_qaoa)."""
    med_rt = {}
    for n in n_values:
        costs = []
        for inst in dataset[int(n)]:
            psi = run_qaoa(inst["h_diag"], betas, gammas, int(n))
            p = per_instance_success_probability(psi, inst["h_diag"])
            costs.append(1.0 / max(float(p), eps))
        med_rt[int(n)] = float(np.median(costs))
    return med_rt

In [ ]:
# --- One-time setup: benchmark data, classical baselines, training Hamiltonians ---
t0 = time.time()
n_values = list(range(int(CFG["n_min"]), int(CFG["n_max"]) + 1))

print("Building SAT benchmark dataset (cached H_diag)...")
dataset = generate_benchmark_dataset_cached(
    n_values, k=K, r=float(CFG["r"]), test_size=int(CFG["test_size"]), base_seed=int(CFG["seed"]),
)

print("Classical solvers (once)...")
classical = evaluate_classical_once(dataset, CFG)
ws_slope = fit_log2_slope(n_values, [classical["walksat"][n] for n in n_values])
lm_slope = fit_log2_slope(n_values, [classical["walksatlm"][n] for n in n_values])
print(f"  WalkSAT   (p_noise={CFG['walksat_p_noise']})   log2 slope = {ws_slope:.4f}")
print(f"  WalkSATlm (p_noise={CFG['walksatlm_p_noise']}) log2 slope = {lm_slope:.4f}")

print("Training Hamiltonians (once)...")
training_h = generate_training_h_diagonals(
    train_n=int(CFG["train_n"]),
    k=K,
    r=float(CFG["r"]),
    train_size=int(CFG["train_size"]),
    base_seed=int(CFG["seed"]),
    m_sampling="notebook",
)

proxy_h = None
proxy_ns: List[int] = []
if not CFG.get("legacy_objective", False):
    proxy_ns = proxy_n_values_for_training(
        int(CFG["train_n"]),
        proxy_n_span=int(CFG["proxy_n_span"]),
        n_max_cap=int(CFG["n_max"]),
    )
    print(f"Proxy Hamiltonians (v2 slope objective): n in {proxy_ns} ...")
    proxy_h = generate_training_h_diagonals_multi_n(
        proxy_ns,
        k=K,
        r=float(CFG["r"]),
        train_size_per_n=int(CFG["proxy_size_per_n"]),
        base_seed=int(CFG["seed"]) + 77,
        m_sampling="notebook",
    )

print(f"Setup done in {time.time() - t0:.1f}s")

In [ ]:
# --- Depth sweep: train + LR benchmark only (always runs ALL depths in CFG["depths"]) ---
from bm24_run_io import make_run_stem, resolve_bm24_run_output_paths


def _trace_payload(trace_rows):
    return {
        "run_stem": run_stem,
        "config": {k: (list(v) if isinstance(v, range) else v) for k, v in CFG.items()},
        "trace": trace_rows,
    }


def _write_trace_checkpoint(trace_rows):
    payload = _trace_payload(trace_rows)
    out_json = run_paths["json"]
    partial_json = run_paths["partial_json"]
    for path in (partial_json, out_json):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, default=str)
    return out_json


recover_path = CFG.get("recover_from")
if recover_path is not None:
    recover_path = Path(recover_path)
    if not recover_path.is_file():
        raise FileNotFoundError(f"recover_from not found: {recover_path.resolve()}")
    with open(recover_path, encoding="utf-8") as f:
        saved = json.load(f)
    trace = list(saved.get("trace", []))
    run_stem = str(
        saved.get("run_stem")
        or CFG.get("run_stem")
        or recover_path.stem.replace(".partial", "")
    )
    print(f"Recovered {len(trace)} depth(s) from {recover_path.name}")
else:
    trace = []
    run_stem = str(CFG.get("run_stem") or make_run_stem("efficient-scaling"))

run_paths = resolve_bm24_run_output_paths(CFG["output_dir"], run_stem)
print(f"Run id: {run_stem}")
print(f"Output dir: {run_paths['run_dir']}")
depths_to_run = [int(p) for p in CFG["depths"]]
completed = {int(r["depth"]) for r in trace}
remaining = [d for d in depths_to_run if d not in completed]
print(f"Will run {len(depths_to_run)} depths: {depths_to_run[0]} … {depths_to_run[-1]} (no early stop on win)")
if completed:
    print(f"Already done: {sorted(completed)}")
    print(f"Remaining: {remaining}")

if CFG.get("legacy_objective", False):
    _proxy_h, _proxy_ns = None, None
else:
    _proxy_h, _proxy_ns = proxy_h, proxy_ns

prev_deltas = None
prev_med_rt = None
prev_accepted_deltas = None
if trace:
    last = trace[-1]
    prev_deltas = (float(last["delta_gamma"]), float(last["delta_beta"]))
    prev_accepted_deltas = prev_deltas
    med = last.get("median_runtime_per_n")
    if med:
        prev_med_rt = {int(k): float(v) for k, v in med.items()}

_eval_rt_factor = float(
    CFG.get("eval_runtime_regression_factor", DEFAULT_EVAL_RUNTIME_REGRESSION_FACTOR)
)

for depth in remaining:
    depth = int(depth)
    t_depth = time.time()
    print(f"\n{'=' * 60}\nDepth p = {depth}\n{'=' * 60}")

    def _train_at_depth(retry_index: int):
        warm = (
            tuple(prev_accepted_deltas)
            if retry_index > 0 and prev_accepted_deltas is not None
            else (tuple(prev_deltas) if prev_deltas is not None else None)
        )
        skip_grid = bool(CFG["skip_grid"])
        if retry_index > 0:
            skip_grid = False
        perturb = float(CFG["cobyla_perturb_scale"]) * (1.25 ** int(retry_index))
        train_rng = np.random.default_rng(
            int(CFG["seed"]) + 1000 + depth + 10_000 * int(retry_index)
        )
        if CFG.get("legacy_objective", False):
            _, diag_local = train_lr_grid_search_legacy(
                training_h,
                train_n=int(CFG["train_n"]),
                depth=depth,
                skip_grid=skip_grid,
                beta_schedule=str(CFG["lr_beta_schedule"]),
                cobyla_maxiter=int(CFG["cobyla_maxiter"]),
            )
        else:
            _, diag_local = train_lr_grid_search_bm24(
                training_h,
                train_n=int(CFG["train_n"]),
                depth=depth,
                skip_grid=skip_grid,
                initial_deltas=warm,
                skip_grid_if_warm_start=bool(CFG.get("skip_grid_if_warm_start", True))
                and retry_index == 0,
                beta_schedule=str(CFG["lr_beta_schedule"]),
                cobyla_maxiter=int(CFG["cobyla_maxiter"]),
                cobyla_restarts=int(CFG["cobyla_restarts"]),
                cobyla_perturb_scale=perturb,
                grid_top_k=int(CFG["grid_top_k"]),
                proxy_h_by_n=_proxy_h,
                proxy_n_values=_proxy_ns,
                rng=train_rng,
            )
        dg_l = float(diag_local["best_deltas"][0])
        db_l = float(diag_local["best_deltas"][1])
        print(
            f"  trained dg={dg_l:.6f} db={db_l:.6f} "
            f"train_slope_log2={diag_local.get('best_train_slope_log2', float('nan')):.4f} "
            f"mean_p@train_n={diag_local['best_avg_train_p_succ']:.4e}"
        )
        if diag_local.get("anti_regression_applied"):
            print("  (anti-regression: kept warm-start training objective)")
        if diag_local.get("train_rejected"):
            print(
                f"  (collapse guard: {diag_local.get('train_reject_reason', '')}; "
                f"applied={diag_local.get('collapse_guard_applied', False)})"
            )
        return dg_l, db_l, diag_local

    def _evaluate_at_angles(dg_eval: float, db_eval: float):
        betas_e, gammas_e = make_lr_angles(
            dg_eval, db_eval, depth,
            beta_schedule=str(CFG["lr_beta_schedule"]),
            angle_convention="bm24",
        )
        return evaluate_lr_qaoa_depth(dataset, n_values, betas_e, gammas_e)

    te_result = run_train_eval_with_retries(
        train_at_depth=_train_at_depth,
        evaluate_at_angles=_evaluate_at_angles,
        prev_med_rt=prev_med_rt,
        prev_accepted_deltas=prev_accepted_deltas,
        n_min=int(CFG["n_min"]),
        n_max=int(CFG["n_max"]),
        max_retries=int(CFG.get("eval_train_retries", DEFAULT_EVAL_TRAIN_RETRIES)),
        regression_factor=_eval_rt_factor,
    )
    dg, db = te_result["dg"], te_result["db"]
    diag = te_result["diag"]
    med_rt = te_result["med_rt"]
    eval_rejected = bool(te_result["eval_rejected"])
    reject_reason = te_result["eval_reject_reason"]
    trained_dg, trained_db = (
        float(diag.get("trained_deltas", diag["best_deltas"])[0]),
        float(diag.get("trained_deltas", diag["best_deltas"])[1]),
    )
    if not eval_rejected:
        prev_accepted_deltas = (dg, db)
        prev_med_rt = dict(med_rt)

    prev_deltas = prev_accepted_deltas
    lr_slope = fit_log2_slope(n_values, [med_rt[n] for n in n_values])
    beats_ws = bool(np.isfinite(lr_slope) and np.isfinite(ws_slope) and lr_slope < ws_slope)
    beats_lm = bool(np.isfinite(lr_slope) and np.isfinite(lm_slope) and lr_slope < lm_slope)

    row = {
        "depth": depth,
        "delta_gamma": float(dg),
        "delta_beta": float(db),
        "trained_delta_gamma": float(trained_dg),
        "trained_delta_beta": float(trained_db),
        "train_rejected": bool(diag.get("train_rejected", False)),
        "train_reject_reason": diag.get("train_reject_reason"),
        "collapse_guard_applied": bool(diag.get("collapse_guard_applied", False)),
        "eval_rejected": bool(eval_rejected),
        "eval_reject_reason": reject_reason if eval_rejected else None,
        "eval_train_attempts": int(te_result.get("eval_train_attempts", 1)),
        "used_previous_angles": bool(te_result.get("used_previous_angles", False)),
        "training_failed": bool(
            te_result.get("used_previous_angles") or te_result.get("eval_rejected")
        ),
        "training_objective": diag.get(
            "objective_version",
            "legacy-mean-p@train_n" if CFG.get("legacy_objective") else None,
        ),
        "train_score": diag.get("best_train_score"),
        "best_train_slope_log2": diag.get("best_train_slope_log2"),
        "lr_log2_slope": lr_slope,
        "walksat_log2_slope": ws_slope,
        "walksatlm_log2_slope": lm_slope,
        "median_runtime_per_n": {str(n): med_rt[n] for n in n_values},
        "lr_beats_walksat_scaling": beats_ws,
        "lr_beats_walksatlm_scaling": beats_lm,
        "elapsed_s": time.time() - t_depth,
    }
    trace.append(row)
    print(f"  LR log2 slope={lr_slope:.4f}  (WS={ws_slope:.4f} LM={lm_slope:.4f})")
    print(f"  beats WS={beats_ws} beats LM={beats_lm}  elapsed={row['elapsed_s']:.1f}s")
    out_json = _write_trace_checkpoint(trace)
    print(f"  checkpoint -> {out_json.name}")

if not remaining:
    out_json = _write_trace_checkpoint(trace)
    print("Nothing left to run — trace already complete.")

print(f"\nWrote {out_json}")

In [ ]:
# --- Optional: plot from saved JSON only (no re-run) ---
# Set plot_json to a finished .json if you only want the graph.
plot_json = None  # e.g. Path("bm24_runs/05-29_1922-efficient-scaling.json")

if plot_json is not None:
    with open(plot_json, encoding="utf-8") as f:
        saved = json.load(f)
    trace = saved["trace"]
    run_stem = saved.get("run_stem", Path(plot_json).stem)
    if trace and "walksat_log2_slope" in trace[0]:
        ws_slope = trace[0]["walksat_log2_slope"]
        lm_slope = trace[0]["walksatlm_log2_slope"]
    else:
        raise RuntimeError("JSON trace missing classical slopes; run setup cell first")
    print(f"Loaded {len(trace)} depths from {plot_json}")
else:
    print("plot_json is None — run the summary plot cell below after the sweep finishes")

In [ ]:
# --- Summary plot (same y as pipeline scaling-vs-depth) ---
from bm24_run_io import plot_scaling_vs_depth, scaling_plot_headline
from IPython.display import Image, display

win_depths = [
    row["depth"]
    for row in trace
    if row.get("lr_beats_walksat_scaling") and row.get("lr_beats_walksatlm_scaling")
]
if win_depths and not CFG.get("annotate_first_win", False):
    print(
        f"(info) first depth beating both classical slopes: p={min(win_depths)} "
        "(annotate_first_win=False)"
    )
plot_path = plot_scaling_vs_depth(
    trace,
    run_paths["png"],
    settings={**CFG, "k": K},
    headline=scaling_plot_headline(
        "Efficient LR sweep", CFG["eval_axis"], CFG["eval_aggregation"]
    ),
    annotate_first_win=bool(CFG.get("annotate_first_win", False)),
)
display(Image(filename=str(plot_path)))
print(f"Saved {plot_path}")